# 08 — Temporal Anomaly Detection

## Objective
Detect bursts and regime shifts with rolling robust baselines rather than global thresholds.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Rolling volume/value baselines

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data()
def temporal_frame(df,w=14):
 d=df.assign(day=df.purchase_time.dt.floor("D")).groupby("day").agg(transactions=("user_id","size"),value=("purchase_value","sum"),fraud_rate=("class","mean")).reset_index()
 for c in ["transactions","value"]:
  med=d[c].rolling(w,min_periods=5).median();
  mad=(d[c]-med).abs().rolling(w,min_periods=5).median(); 
  d[c+"_score"]=((d[c]-med)/(mad.replace(0,np.nan))).fillna(0).abs()


 d["temporal_score"]=d[["transactions_score","value_score"]].max(axis=1); 
 return d


td=temporal_frame(fraud); display(td.tail(20))

,day,transactions,value,fraud_rate,transactions_score,value_score,temporal_score
330,2015-11-27 00:00:00+00:00,115,4243,0.069565,0.918919,1.073901,1.073901
331,2015-11-28 00:00:00+00:00,98,3481,0.081633,1.153153,1.347410,1.347410
332,2015-11-29 00:00:00+00:00,91,3694,0.054945,1.000000,0.793027,1.000000
333,2015-11-30 00:00:00+00:00,69,2533,0.028986,1.564516,1.553345,1.564516
334,2015-12-01 00:00:00+00:00,88,3306,0.079545,0.806452,0.716765,0.806452
335,2015-12-02 00:00:00+00:00,73,2654,0.082192,0.992126,0.982146,0.992126
336,2015-12-03 00:00:00+00:00,61,2265,0.000000,1.129771,1.020841,1.129771
337,2015-12-04 00:00:00+00:00,43,1626,0.023256,1.525926,1.376526,1.525926
338,2015-12-05 00:00:00+00:00,49,1974,0.081633,1.214815,1.049390,1.214815
339,2015-12-06 00:00:00+00:00,39,1532,0.076923,1.394366,1.235988,1.394366


## 2. Validation ranking

In [2]:
px.line(td,x="day",y=["transactions_score","value_score"],
        title="Rolling temporal signals").show(); 
display(td.sort_values("temporal_score",ascending=False).head(15))

,day,transactions,value,fraud_rate,transactions_score,value_score,temporal_score
13,2015-01-14 00:00:00+00:00,75,2709,0.066667,10.570093,5.425470,10.570093
12,2015-01-13 00:00:00+00:00,164,5927,0.585366,10.279570,5.224654,10.279570
14,2015-01-15 00:00:00+00:00,64,2599,0.031250,9.528926,5.128407,9.528926
15,2015-01-16 00:00:00+00:00,69,2543,0.115942,7.719298,4.116706,7.719298
228,2015-08-17 00:00:00+00:00,576,20627,0.053819,2.517241,6.781109,6.781109
16,2015-01-17 00:00:00+00:00,79,2924,0.075949,6.585366,3.534822,6.585366
235,2015-08-24 00:00:00+00:00,556,20303,0.046763,4.075472,5.476390,5.476390
174,2015-06-24 00:00:00+00:00,590,22044,0.044068,4.800000,2.917563,4.800000
17,2015-01-18 00:00:00+00:00,86,2909,0.023256,4.466667,2.724778,4.466667
162,2015-06-12 00:00:00+00:00,566,20287,0.054770,2.945946,4.293548,4.293548


## 3. Interactive window

In [3]:
w=widgets.IntSlider(value=14,min=5,max=30,description="Window"); out=widgets.Output()
def draw(*_):
 d=temporal_frame(fraud,w.value)
 with out: out.clear_output(); 
 px.line(d,x="day",y="temporal_score",title=f"Temporal score — {w.value} days").show()
w.observe(draw,'value'); display(w,out); draw(); 
td.to_parquet(ART/"temporal_daily_scores.parquet")

IntSlider(value=14, description='Window', max=30, min=5)

Output()